<a href="https://colab.research.google.com/github/ojbaker/DEVOPS/blob/main/Nurse_Mate_API_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nurse Mate Smart Assistant API

This notebook contains the complete FastAPI implementation for the Nurse Mate backend, including the SMART on FHIR R4 endpoints, the NLP intent routing logic and The Code Testing


### Run Server in Notebook
FastAPI normally blocks the thread. To run it inside the Notebook interactively, we apply `nest_asyncio` and start the server using `uvicorn`.

In [6]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
from datetime import datetime, timezone
import uuid
import threading # Import threading

# Ensure nest_asyncio is applied early for Jupyter/Colab compatibility
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

# Initialize FastAPI Application
app = FastAPI(
    title="Nurse Mate Smart Assistant API",
    version="1.0.1",
    description="Intelligent Communication Tool & Clinical Co-Pilot for Nurses"
)

# ===============================================================================================================
# 1. API CONTRACTS (Data Models)
# ===============================================================================================================

class FHIRPatient(BaseModel):
    """Contract for EMR R4 Patient Resource"""
    resourceType: str = "Patient"
    id: str
    name: List[Dict[str, Any]]
    gender: str
    birthDate: str

class FHIRDocumentReference(BaseModel):
    """Contract for EMR R4 Clinical Note / Document Reference"""
    resourceType: str = "DocumentReference"
    status: str = Field("current", description="Required in R4: Status of this document reference")
    subject: Dict[str, Any] = Field(..., description="Reference to the Patient")
    type: Dict[str, Any] = Field(..., description="Document type (e.g., LOINC code)")
    content: List[Dict[str, Any]]
    context: Optional[Dict[str, Any]] = None

class TranscriptInput(BaseModel):
    """Contract for incoming Speech-to-Text processed payload"""
    sessionID: str
    userID: str
    rawText: str

class AssistantResponse(BaseModel):
    """Contract for the Smart Assistant's NLP resolution"""
    intentLabel: str
    confidence: float
    actionTaken: str
    data: Dict[str, Any]

class TaskCreate(BaseModel):
    """Contract for assigning a new task"""
    title: str
    assignedTo: str
    priority: int

class Task(TaskCreate):
    """Contract for a managed task within the system"""
    taskID: str
    status: str
    dueTime: datetime

# ===============================================================================================================
# 2. IN-MEMORY MOCKS & DATABASES
# ===============================================================================================================
MOCK_PATIENTS = {
    "123": FHIRPatient(
        id="123",
        name=[{"family": "Smith", "given": ["Jane"]}],
        gender="female",
        birthDate="1975-06-15"
    )
}
TASKS_DB = []

# ===============================================================================================================
# 3. DOMAIN SERVICES (Business Logic)
# ===============================================================================================================

class NLPEngine:
    @staticmethod
    def classify_intent(text: str) -> str:
        text = text.lower()
        if "vitals" in text or "blood pressure" in text or "assessment" in text:
            return "RECORD_ASSESSMENT"
        elif "lab" in text:
            return "QUERY_LAB_RESULTS"
        elif "task" in text or "delegate" in text:
            return "CREATE_TASK"
        return "GENERAL_COMMUNICATION"

class EMRInterface:
    @staticmethod
    def get_patient(patient_id: str) -> FHIRPatient:
        if patient_id in MOCK_PATIENTS:
            return MOCK_PATIENTS[patient_id]
        raise HTTPException(status_code=404, detail="Patient not found in EMR Sandbox")

    @staticmethod
    def post_note(note: FHIRDocumentReference) -> dict:
        return {"status": "success", "documentID": str(uuid.uuid4())}

# =================================================================================================================
# 4. RESTFUL ENDPOINTS
# =================================================================================================================

@app.post("/api/v1/assistant/process", response_model=AssistantResponse)
def process_input(transcript: TranscriptInput):
    intent = NLPEngine.classify_intent(transcript.rawText)

    action_taken = "None"
    data_payload = {}

    if intent == "RECORD_ASSESSMENT":
        doc = FHIRDocumentReference(
            status="current",
            subject={"reference": "Patient/123"},
            type={"coding": [{"system": "http://loinc.org", "code": "11488-4", "display": "Consult note"}]},
            content=[{"attachment": {"data": "base64encoded_clinical_text==", "contentType": "text/plain"}}]
        )
        res = EMRInterface.post_note(doc)
        action_taken = "Posted Clinical Note to EMR"
        data_payload = res

    elif intent == "QUERY_LAB_RESULTS":
        action_taken = "Queried EMR for Labs"
        data_payload = {"labs": [{"test": "CBC", "result": "Within Normal Limits", "referenceRange": "4.0-10.0"}]}

    return AssistantResponse(
        intentLabel=intent,
        confidence=0.97,
        actionTaken=action_taken,
        data=data_payload
    )

@app.get("/api/v1/emr/Patient/{patient_id}", response_model=FHIRPatient)
def get_patient(patient_id: str):
    return EMRInterface.get_patient(patient_id)

@app.post("/api/v1/tasks", response_model=Task)
def create_task(task: TaskCreate):
    # Compatibility mapping for both Pydantic V1 and V2
    task_data = task.model_dump() if hasattr(task, 'model_dump') else task.dict()

    new_task = Task(
        **task_data,
        taskID=str(uuid.uuid4()),
        status="PENDING",
        dueTime=datetime.now(timezone.utc)
    )
    TASKS_DB.append(new_task)
    return new_task

if __name__ == "__main__":
    import uvicorn

    # Optional Colab/Jupyter compatibility block
    # The nest_asyncio.apply() call has been moved to the top of the cell
    try:
        from google.colab import output
        output.serve_kernel_port_as_iframe(8000, path='/docs')
        print("Google Colab detected. An iframe will be generated below to view the interactive documentation!")
    except ImportError:
        pass

    print("Starting FastAPI server on port 8000... View docs at http://localhost:8000/docs")

    # Run uvicorn in a separate thread to avoid RuntimeError in Colab
    server_thread = threading.Thread(target=uvicorn.run, args=(app,), kwargs={"host": "0.0.0.0", "port": 8000})
    server_thread.start()


<IPython.core.display.Javascript object>

Google Colab detected. An iframe will be generated below to view the interactive documentation!
Starting FastAPI server on port 8000... View docs at http://localhost:8000/docs


## Test Suite

Comprehensive test cases validating API contracts and business logic using pytest and FastAPI's TestClient.

In [8]:
import pytest
from fastapi.testclient import TestClient

# Initialize test client
client = TestClient(app)

def test_get_patient_success():
    """Test UC24: Learn about latest health status (GET FHIR Patient)"""
    response = client.get("/api/v1/emr/Patient/123")

    # Verbose assertion: if this fails, it prints the exact server error
    assert response.status_code == 200, f"Expected 200, got {response.status_code}. Details: {response.text}"

    data = response.json()

    # Contract Verification
    assert data.get("resourceType") == "Patient", f"Invalid resource type: {data.get('resourceType')}"
    assert data.get("id") == "123"

    # FIX: The FHIR 'name' field is a list. We must safely index it with first!
    patient_name_list = data.get("name", [])
    assert len(patient_name_list) > 0, "Patient name list is empty!"
    assert patient_name_list[0].get("family") == "Smith", f"Expected Smith, got {patient_name_list[0].get('family')}"

def test_get_patient_not_found():
    """Test boundary condition for EMR queries"""
    response = client.get("/api/v1/emr/Patient/999")
    assert response.status_code == 404, f"Expected 404, got {response.status_code}. Details: {response.text}"

def test_process_assessment_intent():
    """Test UC06: Record assessment using voice assistant"""
    payload = {
        "sessionID": "sess-001",
        "userID": "nurse-01",
        "rawText": "Patient Jane Smith, vitals stable. Blood pressure 122 over 78. Will notify physician."
    }
    response = client.post("/api/v1/assistant/process", json=payload)
    assert response.status_code == 200, f"Expected 200, got {response.status_code}. Details: {response.text}"

    data = response.json()
    assert data["intentLabel"] == "RECORD_ASSESSMENT"
    assert data["actionTaken"] == "Posted Clinical Note to EMR"
    assert "documentID" in data["data"]

def test_process_lab_query_intent():
    """Test UC11: Query lab database"""
    payload = {
        "sessionID": "sess-002",
        "userID": "nurse-01",
        "rawText": "Nurse Mate, what are the latest lab results?"
    }
    response = client.post("/api/v1/assistant/process", json=payload)
    assert response.status_code == 200, f"Expected 200, got {response.status_code}. Details: {response.text}"

    data = response.json()
    assert data["intentLabel"] == "QUERY_LAB_RESULTS"
    assert data["actionTaken"] == "Queried EMR for Labs"

def test_task_creation():
    """Test UC40: Manager assign task to Nurse"""
    payload = {
        "title": "Administer IV Fluids",
        "assignedTo": "nurse-02",
        "priority": 1
    }
    response = client.post("/api/v1/tasks", json=payload)
    assert response.status_code == 200, f"Expected 200, got {response.status_code}. Details: {response.text}"

    data = response.json()
    assert data["status"] == "PENDING"
    assert "taskID" in data
    assert data["assignedTo"] == "nurse-02"

def test_process_general_intent():
    """Test fallback intent routing for non-clinical chatter"""
    payload = {
        "sessionID": "sess-003",
        "userID": "nurse-01",
        "rawText": "Hello Nurse Mate, just checking the microphone."
    }
    response = client.post("/api/v1/assistant/process", json=payload)
    assert response.status_code == 200, f"Expected 200, got {response.status_code}. Details: {response.text}"

    data = response.json()
    assert data["intentLabel"] == "GENERAL_COMMUNICATION"
    assert data["actionTaken"] == "None"

def test_task_creation_validation_error():
    """Test strict contract validation (missing required fields triggers 422)"""
    payload = {
        "title": "Administer IV Fluids"
        # Missing 'assignedTo' and 'priority' to intentionally trigger Pydantic validation
    }
    response = client.post("/api/v1/tasks", json=payload)
    assert response.status_code == 422, "Expected 422 Unprocessable Entity for missing fields"
    assert "detail" in response.json(), "FastAPI should return validation details"

# Run all tests
if __name__ == "__main__":
    print("Running test suite...")
    test_get_patient_success()
    print("✓ test_get_patient_success passed")

    test_get_patient_not_found()
    print("✓ test_get_patient_not_found passed")

    test_process_assessment_intent()
    print("✓ test_process_assessment_intent passed")

    test_process_lab_query_intent()
    print("✓ test_process_lab_query_intent passed")

    test_task_creation()
    print("✓ test_task_creation passed")

    test_process_general_intent()
    print("✓ test_process_general_intent passed")

    test_task_creation_validation_error()
    print("✓ test_task_creation_validation_error passed")

    print("\nAll tests passed!")

Running test suite...
✓ test_get_patient_success passed
✓ test_get_patient_not_found passed
✓ test_process_assessment_intent passed
✓ test_process_lab_query_intent passed
✓ test_task_creation passed
✓ test_process_general_intent passed
✓ test_task_creation_validation_error passed

All tests passed!


### Asynchronous Endpoint Test Demonstration

When testing asynchronous FastAPI endpoints with `fastapi.testclient.TestClient`, the test functions themselves often look synchronous. This is because `TestClient` manages the asynchronous event loop internally, allowing you to make direct calls (`client.get()`, `client.post()`, etc.) without needing `await` in your test code. The client effectively executes the asynchronous FastAPI application within a synchronous testing context.

In [10]:
# Re-initialize test client (optional, but good practice if in a separate context)
client = TestClient(app)

def test_async_general_communication():
    """Demonstrate testing a general communication (async) endpoint using TestClient"""
    payload = {
        "sessionID": "demo-sess-001",
        "userID": "demo-user-01",
        "rawText": "Hello, how are you today?"
    }
    # Call to an asynchronous endpoint; TestClient handles the async execution
    response = client.post("/api/v1/assistant/process", json=payload)

    assert response.status_code == 200, f"Expected 200, got {response.status_code}. Details: {response.text}"

    data = response.json()
    assert data["intentLabel"] == "GENERAL_COMMUNICATION"
    assert data["actionTaken"] == "None"
    print("✓ test_async_general_communication passed")


# Run the demonstration test
if __name__ == "__main__":
    test_async_general_communication()

✓ test_async_general_communication passed


In [12]:
client = TestClient(app)

def test_task_creation_invalid_priority_type():
    """Test API's response to invalid input type for priority field (expecting 422)"""
    payload = {
        "title": "Follow up with Dr. Smith",
        "assignedTo": "nurse-03",
        "priority": "high"  # Invalid type: expecting int, providing string
    }
    response = client.post("/api/v1/tasks", json=payload)

    assert response.status_code == 422, f"Expected 422 for invalid priority type, got {response.status_code}. Details: {response.text}"
    response_data = response.json()
    assert "detail" in response_data, "Response should contain validation details"
    # Optionally, check for specific error message related to 'priority'
    assert any("unable to parse string as an integer" in error.get("msg", "").lower() for error in response_data.get("detail", [])), \
        f"Expected 'unable to parse string as an integer' error for priority, but got: {response_data.get('detail')}"
    print("✓ test_task_creation_invalid_priority_type passed")

# Run the new test
if __name__ == "__main__":
    test_task_creation_invalid_priority_type()

✓ test_task_creation_invalid_priority_type passed


# Nurse Mate Frontend UI

We will use **Gradio** to create an interactive interface. This UI will allow you to:
1. **View Active Tasks**: Fetch and display tasks from the `TASKS_DB`.
2. **Chat/Voice Assistant**: Send text or simulated voice inputs to the `/api/v1/assistant/process` endpoint.

In [ ]:
!pip install -q gradio

In [13]:
import gradio as gr
import pandas as pd

# We use the existing TestClient to bridge the UI to the local API without needing external URLs
ui_client = TestClient(app)

def process_voice_or_text(text, audio_path):
    # If audio is provided, we simulate a transcription
    # In a real app, you'd call a STT service here
    input_text = text if text else "[Simulated Transcript from Audio]"

    payload = {
        "sessionID": "ui-session-001",
        "userID": "nurse-user",
        "rawText": input_text
    }

    response = ui_client.post("/api/v1/assistant/process", json=payload)
    if response.status_code == 200:
        data = response.json()
        return f"**Intent:** {data['intentLabel']}\n**Action:** {data['actionTaken']}"
    return "Error processing request."

def get_tasks_df():
    # Convert the internal TASKS_DB to a displayable DataFrame
    if not TASKS_DB:
        return pd.DataFrame([{"Status": "No tasks assigned"}])
    return pd.DataFrame([{
        "ID": t.taskID[:8],
        "Title": t.title,
        "Assigned": t.assignedTo,
        "Priority": t.priority,
        "Status": t.status
    } for t in TASKS_DB])

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏥 Nurse Mate Assistant Dashboard")

    with gr.Tab("Assistant & Voice"):
        with gr.Row():
            with gr.Column():
                txt_input = gr.Textbox(label="Type clinical note or query")
                audio_input = gr.Audio(label="Record Voice Input", type="filepath")
                submit_btn = gr.Button("Process Input", variant="primary")
            with gr.Column():
                output_text = gr.Markdown("**Assistant Output will appear here...**")

        submit_btn.click(process_voice_or_text, inputs=[txt_input, audio_input], outputs=output_text)

    with gr.Tab("Task Board"):
        task_table = gr.Dataframe(value=get_tasks_df(), interactive=False)
        refresh_btn = gr.Button("Refresh Tasks")
        refresh_btn.click(get_tasks_df, outputs=task_table)

# Launch the UI
demo.launch(debug=True, inline=True)

/usr/lib/python3.12/ast.py:52: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return compile(source, filename, mode, flags,
/tmp/ipykernel_5688/210472418.py:36: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://903d0648143fea433f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://903d0648143fea433f.gradio.live


### Synthetic Data & PRN Reminders

We are expanding the `MOCK_PATIENTS` database and adding a `PRN_LOG` to track medication effectiveness checks.

In [14]:
from datetime import timedelta

# 1. Load Synthetic Patients
NEW_PATIENTS = {
    "124": FHIRPatient(id="124", name=[{"family": "Doe", "given": ["John"]}], gender="male", birthDate="1982-11-20"),
    "125": FHIRPatient(id="125", name=[{"family": "Garcia", "given": ["Maria"]}], gender="female", birthDate="1960-03-05")
}
MOCK_PATIENTS.update(NEW_PATIENTS)

# 2. PRN Tracking System
PRN_LOG = []

def log_prn_medication(patient_id, med_name):
    administration_time = datetime.now()
    due_time = administration_time + timedelta(minutes=60)
    entry = {
        "patient_id": patient_id,
        "medication": med_name,
        "given_at": administration_time.strftime("%H:%M"),
        "due_at": due_time.strftime("%H:%M"),
        "status": "Waiting"
    }
    PRN_LOG.append(entry)
    return f"Logged {med_name} for Patient {patient_id}. Effectiveness check due at {entry['due_at']}."

In [ ]:
def get_prn_df():
    if not PRN_LOG:
        return pd.DataFrame([{"Status": "No active PRN checks"}])

    current_time = datetime.now()
    df = pd.DataFrame(PRN_LOG)

    # Calculate dynamic status
    def check_status(row):
        due_dt = datetime.strptime(row['due_at'], "%H:%M").replace(
            year=current_time.year, month=current_time.month, day=current_time.day
        )
        if current_time > due_dt:
            return "⚠️ OVERDUE"
        return "Pending"

    df['Live Status'] = df.apply(check_status, axis=1)
    return df

# Update UI to include PRN management with highlighting
with gr.Blocks(theme=gr.themes.Soft()) as demo_v2:
    gr.Markdown("# 🏥 Nurse Mate Advanced Dashboard")

    with gr.Tab("Clinical Assistant"):
        with gr.Row():
            with gr.Column():
                txt_input = gr.Textbox(label="Assistant Query")
                submit_btn = gr.Button("Process")
            with gr.Column():
                output_text = gr.Markdown("Output...")
        submit_btn.click(process_voice_or_text, inputs=[txt_input, gr.State(None)], outputs=output_text)

    with gr.Tab("PRN Effectiveness"):
        with gr.Row():
            p_id = gr.Dropdown(choices=list(MOCK_PATIENTS.keys()), label="Patient ID")
            med = gr.Textbox(label="Medication Name (e.g. Morphine)")
            log_btn = gr.Button("Log PRN Admin")

        gr.Markdown("### Active PRN Effectiveness Monitoring")
        prn_table = gr.Dataframe(value=get_prn_df(), interactive=False)
        refresh_prn = gr.Button("Refresh Status")

        log_btn.click(log_prn_medication, inputs=[p_id, med], outputs=gr.Textbox(label="System Status")).then(get_prn_df, outputs=prn_table)
        refresh_prn.click(get_prn_df, outputs=prn_table)

    with gr.Tab("Patient EMR"):
        gr.JSON(value=MOCK_PATIENTS)

demo_v2.launch(debug=True, inline=True)

/tmp/ipykernel_5688/1318173906.py:21: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo_v2:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b7625adfd10aa2e952.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
